# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AHAAkash/-flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring (Lane 2).**

I'm picking this lane because the starter dataset already shows a large, evidence-backed pool of
candidate pages — content that is both losing visibility *and* still carries real search demand —
which is exactly the kind of "too many candidates, too little reviewer time" problem a ranked
queue is built for. It also matches how FlyRank's own pipeline is scoped (baseline rule →
learned model → ranked queue with reason codes), so I can build on a workflow that already has a
runnable reference implementation (`scripts/01`–`05`) and a documented baseline-vs-model
comparison (`outputs/model_report.md`) to sanity-check my own numbers against. I may still switch
to Lane 4 (CTR/Engagement Opportunity Scoring) later if the position-adjusted CTR gap turns out to
be the stronger signal once I dig into the warehouse — this is a provisional lane, not a locked-in
one.


In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"rows: {len(df):,}")
print(f"clients: {df['client_id'].nunique()}")
print(f"avg pages per client: {len(df) / df['client_id'].nunique():.0f}")


rows: 30,000
clients: 32
avg pages per client: 938


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Given a client's existing content inventory, which pages should a content
reviewer look at first this week to refresh, and how confident should they be in that ranking?

- **Unit of analysis:** one content item (page) — `content_id`, nested inside `client_id`.
- **Decision it improves:** which page(s) a content team spends its limited review/editing hours
  on first, out of an inventory that can run into the thousands of pages per client.
- **Who acts, and how:** a content strategist or SEO editor at a FlyRank client. They open the top
  of my ranked queue, read the reason code(s) attached to each page (e.g. "declining with demand",
  "stale but visible"), and decide whether to refresh, expand, protect, or leave the page alone.
  Nobody is asked to act on a bare score — the reason codes are what make the ranking usable.
- **Output:** a ranked review queue (not a single number) — page id, score, reason code(s), and a
  confidence label (high / medium / low).
- **Cost of a wrong call, both directions:**
  - *False positive* (I flag a page that didn't need attention): a few editor-hours spent on a
    page that wouldn't have moved the needle — real but recoverable cost, since editors can
    triage further using the reason codes before committing full effort.
  - *False negative* (a genuinely declining, high-demand page never surfaces): lost visibility and
    lost clicks/sessions compound silently, and by the time someone notices, the page may have
    fallen further down the results. This is the more expensive error, which is why I'll weight
    recall on high-demand pages and use precision@K (not accuracy) to judge the queue — a reviewer
    only ever looks at the top of the list, not the whole inventory.
- **Why data/ML helps, not just a rule:** a simple threshold rule ("flag if trend is down and
  impressions are high") already exists and is a fine baseline — but it treats every declining
  page the same, ignores how position, freshness, word count, and traffic history interact, and
  can't be checked against an actual held-out outcome. A learned ranking can combine many weak,
  tangled signals into one ordering and can be *validated* (precision@K against a real label) in a
  way a hand-written if-statement cannot.


In [2]:
# Reviewer-capacity context: with ~32 clients and ~940 pages/client on average,
# a human reviewer clearly cannot manually check every page every week —
# which is the concrete reason a ranked, capacity-aware queue (not a full audit) is the right output.
pages_per_client = df.groupby("client_id").size()
print(pages_per_client.describe()[["mean", "min", "50%", "max"]])


mean     937.5
min        3.0
50%      567.0
max     7008.0
dtype: float64


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Number 1: how much of the inventory is plausibly worth reviewing at all
# (declining trend AND enough demand that it's not just noise -- mirrors the
# starter pipeline's own "declining_with_demand" reason code: trend_direction == 'down'
# and impressions_90d >= 100)
declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100))
n_declining = declining_with_demand.sum()
pct_declining = 100 * n_declining / len(df)
print(f"1) declining-with-demand pages: {n_declining:,} of {len(df):,} rows ({pct_declining:.1f}%)")

# Number 2: how much search demand actually sits inside that declining-with-demand slice --
# i.e. is this noise, or does it carry real traffic weight?
impressions_at_risk = df.loc[declining_with_demand, "impressions_90d"].sum()
total_impressions = df["impressions_90d"].sum()
pct_impressions_at_risk = 100 * impressions_at_risk / total_impressions
print(f"2) share of all 90d impressions sitting in declining-with-demand pages: "
      f"{pct_impressions_at_risk:.1f}% ({impressions_at_risk:,.0f} of {total_impressions:,.0f})")

# Number 3: the starter pipeline already ran this exact lane end-to-end and logged honest,
# client-held-out numbers -- reused here (not recomputed) as evidence the lane is learnable,
# not just plausible-sounding. Source: outputs/model_report.md
baseline_precision_at_50 = 0.240
model_precision_at_50 = 0.740
print(f"3) starter pipeline, client-holdout precision@50: baseline rules = "
      f"{baseline_precision_at_50:.2f}, random forest = {model_precision_at_50:.2f} "
      f"(about {round(model_precision_at_50*50)} of the top 50 right vs "
      f"{round(baseline_precision_at_50*50)} for the rule)")


1) declining-with-demand pages: 13,152 of 30,000 rows (43.8%)
2) share of all 90d impressions sitting in declining-with-demand pages: 51.2% (79,887,612 of 156,010,989)
3) starter pipeline, client-holdout precision@50: baseline rules = 0.24, random forest = 0.74 (about 37 of the top 50 right vs 12 for the rule)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work CAN claim, by the end of 7 weeks:**

- **Observed / decision-support**: "this page shows declining visibility alongside real search
  demand, based on trailing-90-day observed signals" — a description of what happened, not a
  guarantee about what happens next.
- **Directional / ranked confidence**: "of the top 50 pages in this queue, roughly X historically
  matched the outcome we're scoring for" (precision@K, measured against a held-out, client-grouped
  test set) — a statement about how good the *ordering* is, not a certainty about any single page.
- **A decision-support tool**: the queue narrows a reviewer's attention from thousands of pages to
  a short, reason-coded shortlist. It recommends *where to look first*, not what to do once someone
  looks.

**What this work will NEVER claim:**

- **No causal proof.** I cannot say "refreshing this page will fix it" or "this page declined
  *because of* X" — the data has no experiment or A/B design behind it, so any causal statement
  would be false confidence dressed as a finding. At most: "pages with these characteristics were
  later reviewed/refreshed and some recovered" — correlation, clearly labeled as such.
- **No Google-algorithm claims.** I will not claim to have found or predicted a Google ranking
  factor. The data shows outcomes (position, clicks, impressions), never Google's internal logic.
- **No claim that "declining" means "broken."** Section 7 of the lane guide is explicit that drops
  can be consolidation, seasonality, or plain noise — I will check magnitude, persistence, and
  minimum volume before calling anything a decline, and say so in the write-up.
- **No treating a single low-volume page's score as certain.** Every score below my chosen minimum
  volume threshold gets flagged low-confidence, not silently dropped or trusted.


In [4]:
# Quick self-check: confirm the columns I'm working with are observable signals / ids only --
# no raw titles, URLs, domains, or client names anywhere in the starter dataset.
text_like = [c for c in df.columns if any(k in c.lower() for k in ["url", "title", "domain", "name", "query"])]
print("columns that could contain raw text/identifiers:", text_like)
print("content_id sample (pseudonymized):", df['content_id'].iloc[0])
print("client_id sample (pseudonymized):", df['client_id'].iloc[0])


columns that could contain raw text/identifiers: []
content_id sample (pseudonymized): content_304f48230142
client_id sample (pseudonymized): client_f369cb89fc


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.